<a href="https://colab.research.google.com/github/hridayrjain/AI-Lab/blob/main/1BM24CS119_Week7_Unification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1BM24CS119

---


Hriday R Jain

In [4]:
import re

# --- 1. Variable Detection Standard ---
def is_variable(x):
    # Standard FOL convention: variables are single uppercase characters
    return isinstance(x, str) and x.strip().isupper() and len(x.strip()) == 1

# --- 2. Format / Conversion Helpers ---
def to_logic_string(expr):
    if not isinstance(expr, list):
        return str(expr)
    if len(expr) == 1:
        return f"{expr[0]}()"
    return f"{expr[0]}({', '.join(to_logic_string(arg) for arg in expr[1:])})"

def format_substitution(sub_map):
    return {var: to_logic_string(val) for var, val in sub_map.items()}

# --- 3. Parsing Logic ---
def parse_input(expr_str):
    expr_str = expr_str.strip().replace(" ", "")
    if "(" not in expr_str:
        return expr_str

    match = re.match(r"^([^(]+)\((.*)\)$", expr_str)
    if not match:
        return expr_str

    predicate = match.group(1)
    args_str = match.group(2)

    args = []
    bracket_level = 0
    current_arg = []

    for char in args_str:
        if char == ',' and bracket_level == 0:
            args.append(parse_input("".join(current_arg)))
            current_arg = []
        else:
            if char == '(':
                bracket_level += 1
            elif char == ')':
                bracket_level -= 1
            current_arg.append(char)

    if current_arg:
        args.append(parse_input("".join(current_arg)))

    return [predicate] + args

# --- 4. Unification Core Engine ---
def unify(expr1, expr2, substitution=None, step=1):
    if substitution is None:
        substitution = {}

    indent = "  " * step
    expr1_str = to_logic_string(expr1)
    expr2_str = to_logic_string(expr2)
    formatted_sub = format_substitution(substitution)

    print(f"{indent}Step {step}: Unifying {expr1_str} and {expr2_str} with current bindings: {formatted_sub}")

    if expr1 == expr2:
        print(f"{indent}-> Already identical. No new bindings needed.")
        return substitution

    if is_variable(expr1):
        res = unify_variable(expr1, expr2, substitution, step)
        print(f"{indent}-> Result of binding variable '{expr1}': {format_substitution(res) if res is not None else 'None'}")
        return res

    if is_variable(expr2):
        res = unify_variable(expr2, expr1, substitution, step)
        print(f"{indent}-> Result of binding variable '{expr2}': {format_substitution(res) if res is not None else 'None'}")
        return res

    if isinstance(expr1, list) and isinstance(expr2, list):
        if len(expr1) != len(expr2):
            print(f"{indent}-> Failed: Arity mismatch between {expr1_str} and {expr2_str}")
            return None
        res = unify_relations(expr1, expr2, substitution, step + 1)
        return res

    print(f"{indent}-> Failed to unify constant/predicate mismatch: {expr1_str} vs {expr2_str}")
    return None

def unify_variable(var, val, substitution, step):
    indent = "  " * step
    if var in substitution:
        val_str = to_logic_string(val)
        sub_val_str = to_logic_string(substitution[var])
        print(f"{indent}Variable '{var}' already bound to '{sub_val_str}'. Unifying '{sub_val_str}' with '{val_str}'.")
        return unify(substitution[var], val, substitution, step + 1)
    if is_variable(val) and val in substitution:
        sub_val_str = to_logic_string(substitution[val])
        print(f"{indent}Value '{val}' is a bound variable containing '{sub_val_str}'. Unifying '{var}' with '{sub_val_str}'.")
        return unify(var, substitution[val], substitution, step + 1)
    if occur_check(var, val, substitution):
        val_str = to_logic_string(val)
        print(f"{indent}Failed: Occur Check triggered. '{var}' occurs inside '{val_str}'.")
        return None

    new_sub = substitution.copy()
    new_sub[var] = val
    return new_sub

def occur_check(var, val, substitution):
    if var == val:
        return True
    if isinstance(val, list):
        return any(occur_check(var, item, substitution) for item in val)
    if is_variable(val) and val in substitution:
        return occur_check(var, substitution[val])
    return False

def unify_relations(relations1, relations2, substitution, step):
    if not relations1:
        return substitution

    first_sub = unify(relations1[0], relations2[0], substitution, step)
    if first_sub is None:
        return None

    remaining1 = [apply_sub(first_sub, item) for item in relations1[1:]]
    remaining2 = [apply_sub(first_sub, item) for item in relations2[1:]]

    return unify_relations(remaining1, remaining2, first_sub, step)

def apply_sub(substitution, expr):
    if is_variable(expr):
        return substitution.get(expr, expr)
    if isinstance(expr, list):
        return [apply_sub(substitution, item) for item in expr]
    return expr

# --- 5. Interactive Input Prompts ---
try:
    expression_1 = input("Enter Expression 1 (e.g. p(b, X, f(g(Z)))): ")
    expression_2 = input("Enter Expression 2 (e.g. p(Z, f(Y), f(Y))): ")

    parsed_expr1 = parse_input(expression_1)
    parsed_expr2 = parse_input(expression_2)

    print("\nParsed Expression 1:", to_logic_string(parsed_expr1))
    print("Parsed Expression 2:", to_logic_string(parsed_expr2))
    print("\n--- Starting Unification Process ---")

    unify_result = unify(parsed_expr1, parsed_expr2)

    print("------------------------------------")
    if unify_result is not None:
        print("unified successfully")
        print("Final Substitution Map:", format_substitution(unify_result))
    else:
        print("unification failed")
except Exception as e:
    print(f"An error occurred during execution: {e}")

Enter Expression 1 (e.g. p(b, X, f(g(Z)))): p(b, X, f(g(Z)))
Enter Expression 2 (e.g. p(Z, f(Y), f(Y))): p(Z, f(Y), f(Y))

Parsed Expression 1: p(b, X, f(g(Z)))
Parsed Expression 2: p(Z, f(Y), f(Y))

--- Starting Unification Process ---
  Step 1: Unifying p(b, X, f(g(Z))) and p(Z, f(Y), f(Y)) with current bindings: {}
    Step 2: Unifying p and p with current bindings: {}
    -> Already identical. No new bindings needed.
    Step 2: Unifying b and Z with current bindings: {}
    -> Result of binding variable 'Z': {'Z': 'b'}
    Step 2: Unifying X and f(Y) with current bindings: {'Z': 'b'}
    -> Result of binding variable 'X': {'Z': 'b', 'X': 'f(Y)'}
    Step 2: Unifying f(g(b)) and f(Y) with current bindings: {'Z': 'b', 'X': 'f(Y)'}
      Step 3: Unifying f and f with current bindings: {'Z': 'b', 'X': 'f(Y)'}
      -> Already identical. No new bindings needed.
      Step 3: Unifying g(b) and Y with current bindings: {'Z': 'b', 'X': 'f(Y)'}
      -> Result of binding variable 'Y': {'Z'